# Месяц операций в Spark

Один кусок `final_df` за июль 2026: число операций, оборот и интерчейндж.
Таблица `ods_alpha.scd1_trx` читается библиотекой ACID, как в рабочем DAG.
В DRP ничего не пишется.

Ячейки идут сверху вниз. Логин и пароль спросят во второй ячейке.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
for _name in ("py4j", "py4j.java_gateway", "pyspark"):
    logging.getLogger(_name).setLevel(logging.ERROR)

REPORT_MONTH = "2026-07"
print("month", REPORT_MONTH)

In [ ]:
import getpass
import os

from pyspark.sql import SparkSession
from rail_connectors.connection import connect

keytab_path = "/home/jovyan/test_requests/tech.keytab"
os.makedirs(os.path.dirname(keytab_path), exist_ok=True)

LAKE_USER = input("Логин Impala: ").strip()
LAKE_PASSWORD = getpass.getpass("Пароль Impala: ")
if not LAKE_USER or not LAKE_PASSWORD:
    raise RuntimeError("Логин и пароль нужны оба")

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    driver_args={"tez.queue.name": "ai"},
    kerberos={
        "keytab_path": keytab_path,
        "use_credentials": True,
        "update_keytab": True,
    },
    user_params={"user_name": LAKE_USER, "password": LAKE_PASSWORD},
)
imp._init_connection()
with imp:
    imp.execute("SELECT 1")
print("Impala ok")

if "SPARK_K8S_CONF_DIR" in os.environ:
    os.environ["SPARK_CONF_DIR"] = os.environ["SPARK_K8S_CONF_DIR"]

log4j_path = "/tmp/acq_spark_log4j2.properties"
with open(log4j_path, "w", encoding="utf-8") as log4j_file:
    log4j_file.write(
        "\n".join([
            "status = error",
            "name = AcqSpark",
            "appender.console.type = Console",
            "appender.console.name = console",
            "appender.console.target = SYSTEM_ERR",
            "appender.console.layout.type = PatternLayout",
            "appender.console.layout.pattern = %p %c{1}: %m%n",
            "rootLogger.level = error",
            "rootLogger.appenderRef.stdout.ref = console",
        ])
    )
os.environ["SPARK_SUBMIT_OPTS"] = f"-Dlog4j2.configurationFile=file:{log4j_path}"

try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .appName("acq-month-trx")
    .master("yarn")
    .config("spark.yarn.queue", "ai")
    .config("spark.yarn.stagingDir", "hdfs:///tmp")
    .config("spark.yarn.maxAppAttempts", "1")
    .config("spark.dynamicAllocation.enabled", "false")
    .config("spark.executor.instances", "1")
    .config("spark.executor.memory", "20g")
    .config("spark.executor.cores", "5")
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.submit.deployMode", "client")
    .config("spark.log.level", "ERROR")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.orc.filterPushdown", "false")
    .config("spark.sql.orc.aggregatePushdown", "false")
    .config(
        "spark.jars",
        "hdfs:///apps/spark-acid-lib/spark-acid-3.5.2-assembly-0.6.0.jar",
    )
    .config(
        "spark.executor.extraClassPath",
        "hdfs:///apps/spark-acid-lib/spark-acid-3.5.2-assembly-0.6.0.jar",
    )
    .config(
        "spark.sql.extensions",
        "com.qubole.spark.hiveacid.HiveAcidAutoConvertExtension",
    )
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

## Июль: операции, оборот, интерчейндж

Фильтры те же, что в секции операций `final_df`: SA, S01, не возврат, есть терминал, месяц по `d_trx_orig`.

In [ ]:
month_start = f"{REPORT_MONTH}-01"

part = spark.sql(f"""
SELECT
  count(*) AS trx_cnt,
  sum(cast(t.n_amt_src AS double)) AS trx_sum,
  sum(cast(ti.n_amt_fee AS double)) AS int_component
FROM ods_alpha.scd1_trx t
LEFT JOIN ods_alpha.scd1_trx_int ti ON ti.n_trx = t.n_trx
WHERE t.c_trx_class = 'SA'
  AND t.c_trx_type = 'S01'
  AND coalesce(t.cf_trx_stat, '') <> 'R'
  AND t.c_nter IS NOT NULL
  AND trunc(to_date(to_timestamp(t.d_trx_orig, 'yyyy-MM-dd HH:mm:ss')), 'MM')
      = to_date('{month_start}')
""")
part.persist()
part.show()

trx_cnt = part.collect()[0]["trx_cnt"]
if trx_cnt is None or int(trx_cnt) == 0:
    raise RuntimeError(
        "За месяц 0 операций. Библиотека ACID не подхватилась "
        "или фильтр месяца ничего не оставил."
    )
print("rows ok", int(trx_cnt))